# LangSmith & AgentOps

| Part | Tool | What you'll do |
|---|---|---|
| 0 | — | Install packages, configure API keys |
| A | **LangSmith** | Auto-trace a LangChain chain, `@traceable` on raw Python functions, use ChatGoogleGenerativeAI directly, pull traces back programmatically, run a minimal eval |
| B | **AgentOps** | `session`/`agent`/`operation` decorators, 2-line CrewAI integration, session replay |
| C | **Both together** | One CrewAI crew traced by AgentOps (agent-level orchestration) *and* LangSmith (LLM-call-level reasoning) at the same time |

### Prerequisites — get these before running

| Need | Where |
|---|---|
| LangSmith account + API key | [smith.langchain.com](https://smith.langchain.com) → Settings → API Keys |
| AgentOps account + API key | [app.agentops.ai/settings/projects](https://app.agentops.ai/settings/projects) |
| Google API key | Google AI Studio — this notebook uses Gemini as the LLM throughout |

> **Run this in a fresh virtualenv or a new Colab runtime.** This stack has real inter-package version sensitivity (`opentelemetry-*`, `packaging`, `websockets`) — installing it alongside a lot of other pre-existing packages can trigger resolver conflicts.

In [ ]:
# Fresh environment recommended — see note above.
!pip install -qU langsmith langchain langchain-anthropic anthropic agentops crewai python-dotenv langchain-google-genai google-generativeai

## 0. Configure API Keys

`getpass` keeps keys out of the notebook's saved output/history. Swap for a `.env` file + `python-dotenv` if you prefer — either way, **never hardcode keys in a cell you might commit or share.**

A naming note: LangSmith's current env var is `LANGSMITH_TRACING` (the older `LANGCHAIN_TRACING_V2` name still works for backward compatibility, but new docs use the `LANGSMITH_*` prefix — that's what we use below).


In [ ]:
import os
from getpass import getpass

def set_env(var: str, prompt: str):
    if not os.environ.get(var):
        os.environ[var] = getpass(prompt)

# --- LangSmith ---
os.environ["LANGSMITH_TRACING"] = "true"                       # turns tracing on globally
set_env("LANGSMITH_API_KEY", "LangSmith API key: ")
os.environ["LANGSMITH_PROJECT"] = "langsmith-agentops-demo"     # groups these runs in the UI
# os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"  # uncomment for EU tenants

# --- AgentOps ---
set_env("AGENTOPS_API_KEY", "AgentOps API key: ")

# --- LLM provider (Gemini, used throughout this notebook) ---
set_env("GOOGLE_API_KEY", "Google API Key: ")

print("Environment configured.")

## Part A — LangSmith

LangSmith traces at three levels of increasing manual effort: **zero-code** (LangChain/LangGraph runs are captured automatically once tracing is on), **one decorator** (`@traceable` on any Python function), and **one wrapper call** (`wrap_anthropic`/`wrap_openai` around a raw client). Most real apps end up using all three in different places.

### A1. Zero-code tracing — LangChain chains are captured automatically

No decorator needed here — `LANGSMITH_TRACING=true` is doing all the work.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=0.3)

prompt = ChatPromptTemplate.from_template(
    "Explain {concept} in exactly two sentences, for a data science audience."
)
chain = prompt | llm | StrOutputParser()

result = chain.invoke(
    {"concept": "Splink probabilistic record linkage"},
    config={
        "tags": ["demo", "part-a"],
        "metadata": {"notebook_section": "A1-lcel-autotrace"},
    },
)
print(result)

# Open https://smith.langchain.com -> project "langsmith-agentops-demo" to see this
# run's full input/output, latency, and token count, no extra code required.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=0.3)

prompt = ChatPromptTemplate.from_template(
    "Explain {concept} in exactly two sentences, for a data science audience."
)
chain = prompt | llm | StrOutputParser()

result = chain.invoke(
    {"concept": "Splink probabilistic record linkage"},
    config={
        "tags": ["demo", "part-a"],
        "metadata": {"notebook_section": "A1-lcel-autotrace"},
    },
)
print(result)

# Open https://smith.langchain.com -> project "langsmith-agentops-demo" to see this
# run's full input/output, latency, and token count, no extra code required.

### A2. `@traceable` on plain Python functions with LangChain's Google GenAI integration

Use this when your pipeline **isn't** built with LangChain — e.g. a hand-rolled RAG step. `@traceable` groups arbitrary functions — including nested ones — into a single trace tree while utilizing the standard `ChatGoogleGenerativeAI` model wrapper.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langsmith import traceable

# Initialize the LangChain Google GenAI wrapper
llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=0.3)

@traceable(run_type="retriever", name="retrieve_context")
def retrieve_context(query: str) -> str:
    # Stand-in for a real retrieval step
    return f"[mocked context relevant to: {query}]"

@traceable(run_type="chain", name="answer_with_context")
def answer_with_context(query: str) -> str:
    context = retrieve_context(query)
    # Call the LangChain model directly inside our traceable function
    response = llm.invoke(f"Context: {context}\n\nQuestion: {query}")
    return response.content

answer = answer_with_context("How does deterministic vs probabilistic matching differ?")
print(answer)

### A3. Pulling traces back out programmatically

Useful for building your own dashboards, cost-tracking scripts, or CI checks on latency/quality — you don't have to live in the web UI.


In [ ]:
from langsmith import Client

ls_client = Client()

runs = list(
    ls_client.list_runs(
        project_name=os.environ["LANGSMITH_PROJECT"],
        limit=5,
        is_root=True,  # top-level runs only, not every nested span
    )
)

for run in runs:
    latency = None
    if run.end_time and run.start_time:
        latency = (run.end_time - run.start_time).total_seconds()
    print(f"{run.name:<25} status={run.status:<10} latency={latency}s  url={run.url if hasattr(run, 'url') else ls_client.get_run_url(run)}")


### A4. A minimal evaluation loop

Tracing tells you *what happened*; evaluation tells you *whether it was good*. This is the smallest possible version: a 2-example dataset and one custom scoring function, run through `evaluate()`.


In [ ]:
from langsmith.evaluation import evaluate
import google.generativeai as genai

# 1. A tiny reference dataset
dataset_name = "gfg-gemini-series-qa-demo"
existing = {d.name for d in ls_client.list_datasets(dataset_name=dataset_name)}
if dataset_name not in existing:
    dataset = ls_client.create_dataset(dataset_name)
    ls_client.create_examples(
        inputs=[
            {"question": "What is Gemini Pro best suited for?"},
            {"question": "What does the AgentOps SDK track?"},
        ],
        outputs=[
            {"answer": "General-purpose multimodal tasks, reasoning, and standard agentic workflows."},
            {"answer": "Agent sessions, tool calls, decision points, and per-agent cost."},
        ],
        dataset_id=dataset.id,
    )

genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-pro")

# 2. The thing being evaluated
def answer_question(inputs: dict) -> dict:
    resp = model.generate_content(inputs["question"])
    return {"answer": resp.text}

# 3. A simple keyword-overlap scorer
def keyword_overlap(run, example) -> dict:
    predicted = run.outputs["answer"].lower()
    reference = example.outputs["answer"].lower()
    reference_words = set(reference.split())
    hit = sum(1 for w in reference_words if w in predicted) / max(len(reference_words), 1)
    return {"key": "keyword_overlap", "score": hit}

results = evaluate(
    answer_question,
    data=dataset_name,
    evaluators=[keyword_overlap],
    experiment_prefix="gemini-pro-baseline",
)
print("Evaluation experiment created — view the side-by-side comparison in the LangSmith UI.")

## Part B — AgentOps

AgentOps is built around **sessions**: a session is the root span for one full agent run, and everything inside it (agents, operations, tool calls, LLM calls) nests underneath. Three ways in, from most manual to most automatic:


In [ ]:
import agentops
import google.generativeai as genai
from agentops.sdk.decorators import session, agent, operation

agentops.init()  # picks up AGENTOPS_API_KEY from the environment
genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-pro")

@agent
class ResearchAgent:
    def __init__(self, name: str):
        self.name = name

    @operation
    def summarize(self, topic: str) -> str:
        resp = model.generate_content(f"Summarize {topic} in one sentence.")
        return resp.text

@session
def run_research_session(topic: str) -> str:
    researcher = ResearchAgent("research-agent")
    return researcher.summarize(topic)

output = run_research_session("hybrid deterministic-probabilistic record linkage")
print(output)

### B1. CrewAI — observability in (about) two lines

Modern CrewAI (>= 0.105) fires internal lifecycle events (`CrewKickoffStartedEvent`, `CrewKickoffCompletedEvent`, etc.) that AgentOps listens for automatically the moment both packages are installed. `agentops.init()` before your crew runs is the only line you strictly need to add.


In [ ]:
from crewai import Agent, Task, Crew, LLM

# agentops.init() was already called above — CrewAI's built-in event listener
# picks it up automatically for every crew you kick off from here on.

crew_llm = LLM(model="gemini-pro", temperature=0.5)

researcher = Agent(
    role="Senior Data Science Researcher",
    goal="Give accurate, concise answers about maternal & child health data pipelines",
    backstory="An expert in record linkage and healthcare data quality.",
    llm=crew_llm,
    verbose=True,
)

research_task = Task(
    description="In 2-3 sentences, explain why deterministic matching alone fails on messy patient records.",
    expected_output="A concise, technically accurate explanation.",
    agent=researcher,
)

crew = Crew(agents=[researcher], tasks=[research_task], verbose=True)
result = crew.kickoff()
print(result)

### B2. Where to actually look

After a run, AgentOps prints a direct link to that session's page. The dashboard view (`app.agentops.ai/drilldown`) gives you:
- A timeline of every agent, operation, and LLM call, in order
- **Time-travel replay** — step through the run as it happened
- Per-agent cost and token attribution (which agent in a multi-agent crew is expensive)
- Errors and any flagged prompt-injection attempts, end to end


## Part C — Using both at the same time

These two tools aren't really competing for the same job. **LangSmith** is strongest at the LLM-call level: exact prompts, exact completions, token-by-token cost, and evaluation of output *quality*. **AgentOps** is strongest at the agent-orchestration level: which agent did what, in what order, and where a multi-agent run went sideways. Running both simultaneously costs you nothing extra in code — both are driven by environment variables and an `init()`/global-flag call, so they trace independently off the same underlying LangChain/CrewAI calls.

In [ ]:
# Both LANGSMITH_TRACING=true and agentops.init() are already active from earlier cells.
# This single crew run will now show up in BOTH dashboards, from two different angles.

from crewai import Agent, Task, Crew, LLM

dual_traced_llm = LLM(model="gemini-pro", temperature=0.4)

dedup_agent = Agent(
    role="Record Deduplication Reviewer",
    goal="Decide whether two patient records likely refer to the same person",
    backstory="A meticulous reviewer trained on the Splink probabilistic matching pipeline.",
    llm=dual_traced_llm,
    verbose=True,
)

review_task = Task(
    description=(
        "Record A: name='Anita Kumar', dob='1990-03-14', phone='9876500001'. "
        "Record B: name='Anita Kumarr', dob='1990-03-14', phone='9876500001'. "
        "Are these the same person? Answer in one sentence with your confidence level."
    ),
    expected_output="A one-sentence match/no-match decision with a confidence level.",
    agent=dedup_agent,
)

dual_crew = Crew(agents=[dedup_agent], tasks=[review_task], verbose=True)
dual_result = dual_crew.kickoff()
print(dual_result)

# Check app.agentops.ai/drilldown for the agent-level session replay + cost attribution.
# Check smith.langchain.com for the exact prompt/completion and token usage on the Gemini call inside it.

> **A practical note on overhead:** independent benchmarking has found AgentOps and Langfuse-style SDKs add roughly 12–15% latency overhead per call, versus ~5% for the lightest OTel-native instrumentation. Fine for development and most production traffic; worth measuring directly if you're latency-critical.


## Summary

| | LangSmith | AgentOps |
|---|---|---|
| **Unit of observation** | LLM call / chain run | Agent session |
| **Best for** | Prompt/output debugging, cost per call, quality evals | Multi-agent orchestration debugging, session replay, per-agent cost |
| **Enable with** | `LANGSMITH_TRACING=true` + API key (zero code for LangChain) | `agentops.init()` (auto-hooks CrewAI/AutoGen/OpenAI Agents SDK) |
| **CrewAI integration** | Traces the underlying LLM calls if the agent's `LLM` is LangChain-backed | Native, event-based, ~2 lines |
| **Evaluation** | Built-in (`evaluate()`, datasets, custom + LLM-judge scorers) | Not the focus — pair with LangSmith or Confident AI for this |